In [267]:
%load_ext autoreload
%autoreload 2

import pickle
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import glob
from numba import njit, types
from numba.typed import List

from pccross import *

d_dir = './data(mini)'

base_ym = list(range(202401, 202410))
                         
base_ym.sort()
base_ym_begin = base_ym[0]
base_ym_end = base_ym[-1]

opt_atm = pd.DataFrame()
copt = pd.DataFrame()
popt = pd.DataFrame()
fut = pd.DataFrame()
idx = pd.DataFrame()

for base_ym_i in base_ym:
    print(base_ym_i)
    [opt_atm_i, copt_i, popt_i, fut_i] = get_opt_atm(base_ym_i, d_dir)
    opt_atm = pd.concat([opt_atm, opt_atm_i])
    copt = pd.concat([copt, copt_i])
    popt = pd.concat([popt, popt_i])
    fut = pd.concat([fut, fut_i])
    
opt_atm.to_csv("ATM_%d_%d.csv" % (base_ym[0], base_ym[-1]), index=False)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
202401
202402
202403
202404
202405
202406
202407
202408
202409


In [268]:
import time

dt_strtg_begin_buf = 0
dt_strg_end_buf = 0 
dp_lc = 10
dp_pt = 10
unit = 2.5
dp_pt_buf = 0
dt_react_obs = 60
dt_react_hold = 60

In [269]:
def perf_(dt_strtg_begin_buf, dt_strg_end_buf, dp_lc, dp_pt, unit, dp_pt_buf, dt_react_obs, dt_react_hold, opt_atm):
        
    dt_strg_end_buf = 10*60 + dt_strg_end_buf    #전략 종료 시간 버퍼

    ent_dp = 0.5         # 진입 가격 변동
    clr_dp = 0.5         # 대응 가격 변동

    dp_lc = np.ceil(max(0, dp_lc)/unit) * unit
    dp_pt = np.ceil(max(0, dp_pt)/unit) * unit
    dp_pt_buf = min(dp_pt, (np.ceil(dp_pt_buf)/unit) * unit)
     
    #atm 자료 시간 오름 차순으로 정렬
    opt_atm = opt_atm.copy()

    opt_atm['DATE'] = pd.to_datetime(opt_atm['DATE'], format = '%Y%m%d')
    opt_atm['TIME'] = pd.to_datetime(opt_atm['TIME'], format = '%H:%M:%S').dt.time

    opt_atm = opt_atm.sort_values(by = ['DATE', 'TIME'], ascending=[True, True]);

    #A. 장 시작/종료 시간 추출
    mkt_tm_o = opt_atm.groupby(['DATE'])[['DATE', 'TIME']].head(1);
    mkt_tm_c = opt_atm.groupby(['DATE'])[['DATE', 'TIME']].tail(1);
    mkt_tm = pd.merge(mkt_tm_o, mkt_tm_c, how = 'inner', on = ['DATE'], suffixes=['_O', '_C']);

    strtgy_opt = pd.merge(opt_atm, mkt_tm, how = 'inner', on = ['DATE']);

    #Z. 전략 실행
    # atm_strk_cur
    # win_opt
    #손익
    tot_pl = 0
    fee = 0
    fee_rt = 0.003/100
    exec = pd.DataFrame(columns=['DATE', 'TIME', 'POS_TYPE', 'TR', 'PL', 'FEE'])

    #position
    pos = 0
    pos_type = ''

    pr_ent = None # 진입가격
    react_obs_tm = datetime.min
    react_hold_tm = datetime.min    
    pr_h = -np.inf
    pr_l = np.inf

    for idx, atm_i in strtgy_opt.iterrows():
        
        tm_i = datetime.combine(atm_i['DATE'], atm_i['TIME'])
        pr_i = atm_i['FUT']
        pr_o_i = atm_i['FUT_O']
        pr_h_i = atm_i['FUT_H']
        pr_l_i = atm_i['FUT_L']
        
        #장 시작 후, 일정 시간은 거래 skip
        mkt_o_tm = datetime.combine(atm_i['DATE'], atm_i['TIME_O']) 
        if tm_i < mkt_o_tm + timedelta(seconds=dt_strtg_begin_buf):
            continue
        
        #장 종료 전, 청산
        mkt_c_tm = datetime.combine(atm_i['DATE'], atm_i['TIME_C']) 
        if pos != 0 and mkt_c_tm - timedelta(seconds=dt_strg_end_buf) <= tm_i:
            
            pl_clr_i = np.sign(pos) * (pr_i - pr_ent)          
            fee_i = pr_i * fee_rt
            tot_pl = tot_pl + pl_clr_i - fee_i    
            
            pos = 0
            exec_i = pd.DataFrame([{'DATE' : tm_i.date(), 'TIME' : tm_i.time(), 'POS_TYPE' : pos_type, 'POS' : pos, 'TR' : 'CLR', 'PL' : pl_clr_i, 'FEE' : fee_i}])
            if len(exec) == 0:
                exec = exec_i
            else:
                exec = pd.concat([exec, exec_i])
            
            #초기화
            pos = 0              
            pos_type = ''
            
            #장종료 전 청산 시, 대응관찰/유보시간 초기화
            react_obs_tm = datetime.min
            react_hold_tm = datetime.min            
            pr_ent = None # 진입가격
            pr_h = -np.inf
            pr_l = np.inf
            
            continue
        
        #장 종료 전, 일정 시간은 거래 skip 
        if mkt_c_tm - timedelta(seconds=dt_strg_end_buf) <= tm_i:
            continue
                
        pr_h = max(pr_h, pr_h_i)
        pr_l = min(pr_l, pr_l_i)
        
        opt_dmnt_prv_i = atm_i['OPT_DMNT_PRV']
        opt_dmnt_i = atm_i['OPT_DMNT']
        
        #1.1 손절 가격대이면, 손절(대응시간 변경, 가격 초기화, 손익 반영)
        pr_lc_i = 0
        if (pos > 0 and pr_l <= (pr_ent - dp_lc)):
            pr_lc_i = min(pr_ent - dp_lc, pr_o_i)
        elif (pos < 0 and (pr_ent + dp_lc) <= pr_h):
            pr_lc_i = max(pr_ent + dp_lc, pr_o_i)
            
        if pr_lc_i != 0: 
            
            pl_lc_i = np.sign(pos) * (pr_lc_i - pr_ent)          
            fee_i = pr_lc_i * fee_rt
            tot_pl = tot_pl + pl_lc_i - fee_i
            pos = 0              

            exec_i = pd.DataFrame([{'DATE' : tm_i.date(), 'TIME' : tm_i.time(), 'POS_TYPE' : pos_type, 'POS' : pos, 'TR' : 'LC', 'PL' : pl_lc_i, 'FEE' : fee_i}])
            if len(exec) == 0:
                exec = exec_i
            else:
                exec = pd.concat([exec, exec_i])

            #손절 시, 대응관찰/유보시간 초기화
            react_obs_tm = tm_i + timedelta(seconds = dt_react_obs)
            react_hold_tm = tm_i + timedelta(seconds = dt_react_hold)
            
            pr_ent = pr_lc_i
            pr_h = pr_lc_i
            pr_l = pr_lc_i
            
            #초기화
            pos_type = ''
            
            continue    
        
        #1.2 익절 구간이면
        pr_pt_i = 0
        if (pos > 0 and pr_h >= pr_ent + dp_pt):
            pr_pt_i = max(pr_h, pr_ent + dp_pt) - dp_pt_buf
            
        elif (pos < 0 and pr_l <= pr_ent - dp_pt):
            pr_pt_i = min(pr_h, pr_ent - dp_pt) + dp_pt_buf
        
        #A. 익절 구간이고, 현재 가격이 익절 실행 가격이면
        if pr_pt_i != 0 and ((0 < pos and pr_l_i <= pr_pt_i) or (pos < 0 and pr_pt_i <= pr_h_i )) :
            pl_pt_i = np.sign(pos) * (pr_pt_i - pr_ent)          
            fee_i = pr_pt_i * fee_rt
            tot_pl = tot_pl + pl_pt_i - fee_i
            
            pos = 0              
            exec_i = pd.DataFrame([{'DATE' : tm_i.date(), 'TIME' : tm_i.time(), 'POS_TYPE' : pos_type, 'POS' : pos, 'TR' : 'PT', 'PL' : pl_pt_i, 'FEE' : fee_i}])
            if len(exec) == 0:
                exec = exec_i
            else:
                exec = pd.concat([exec, exec_i])

            #익절 시, 대응관찰/유보시간 초기화
            react_obs_tm = datetime.min
            react_hold_tm = datetime.min
            
            pr_ent = pr_pt_i
            pr_h = pr_pt_i
            pr_l = pr_pt_i
            
            #초기화
            pos_type = ''
            continue    
            
        #2.1 대응유보 시간 이내이면, 
        if tm_i < react_hold_tm:
            #아무 작업하지 않음    
            continue
        
        #2.2 대응유보시간 종료했으면 대응유보시간 초기화
        react_hold_tm = datetime.min  
        
        #3.1 대응관찰시간 이내이면 이후
        if tm_i < react_obs_tm:
            #아무 작업하지 않음    
            continue
        
        #3.2 대응관찰 시간 이후, PUT/CALL Cross가 발생했다면, 대응관찰 시작
        react_obs_tm = datetime.min

        if (opt_dmnt_prv_i != opt_dmnt_i) \
            or (pos != 0 and ((pos_type != 'S' and opt_dmnt_i == 'P') or (pos_type != 'L' and opt_dmnt_i == 'C'))) :
                
            if opt_dmnt_i == 'P':
                pos_type = 'S'            
            elif opt_dmnt_i == 'C':
                pos_type = 'L'            
            
            #대응관찰 종료 시간 설정 후, 관찰 시작
            react_obs_tm = tm_i + timedelta(seconds = dt_react_obs)

        
        #3.3 대응관찰시간 종료됐으면, 대응관찰시간 초과 후 옵션크로스 신호의 진위여부에 따라서 대응
                
        #대응관찰시간 초기화
        react_obs_tm = datetime.min
                    
        pl_i = 0   
        
        # 대응관찰 전/후 PUT 우세가 유지되고 있다면
        if pos_type == 'S' and opt_dmnt_i == 'P':
            
            #현재 SHORT 포지션이면, 
            if pos < 0:
                #아무 작업하지 않음
                continue
            
            #현재 LONG 포지션이면, 
            if pos > 0:
                #매도를 통한 손익 실현
                pos = 0
                pos_type = ''
                
                pl_i = (pr_i - pr_ent)
                fee_i = pr_i * fee_rt
                tot_pl = tot_pl + pl_i - fee_i

                exec_i = pd.DataFrame([{'DATE' : tm_i.date(), 'TIME' : tm_i.time(), 'POS_TYPE' : pos_type, 'POS' : pos, 'TR' : 'CLR', 'PL' : pl_i, 'FEE' : fee_i}])
                if len(exec) == 0:
                    exec = exec_i
                else:
                    exec = pd.concat([exec, exec_i])
                
            #SHORT 포지션 진입
            pos = -1
            pos_type = 'S'
            
            fee_i = pr_i * fee_rt
            tot_pl = tot_pl - fee_i
            exec_i = pd.DataFrame([{'DATE' : tm_i.date(), 'TIME' : tm_i.time(), 'POS_TYPE' : pos_type, 'POS' : pos, 'TR' : 'ES', 'PL' : 0, 'FEE' : fee_i}])            
            if len(exec) == 0:
                exec = exec_i
            else:
                exec = pd.concat([exec, exec_i])
            
            react_hold_tm = tm_i + timedelta(seconds = dt_react_hold)
            pr_ent = pr_i
            pr_h = pr_i
            pr_l = pr_i                     
                    
            continue
        
        # 대응관찰 전/후 CALL 우세가 유지되고 있다면
        if pos_type == 'L' and opt_dmnt_i == 'C':
        
            #현재 LONG 포지션이면, 
            if pos > 0:
                #아무 작업하지 않음
                continue
            
            #현재 SHORT 포지션이면, 
            if pos < 0:
                #매수를 통한 손익 실현
                pos = 0
                pos_type = ''
                
                pl_i = (pr_ent - pr_i)                
                fee_i = pr_i * fee_rt
                tot_pl = tot_pl + pl_i - fee_i
                
                exec_i = pd.DataFrame([{'DATE' : tm_i.date(), 'TIME' : tm_i.time(), 'POS_TYPE' : pos_type, 'POS' : pos, 'TR' : 'CLR', 'PL' : pl_i, 'FEE' : fee_i}])
                if len(exec) == 0:
                    exec = exec_i
                else:
                    exec = pd.concat([exec, exec_i])

            #LONG 포지션 진입
            pos = 1
            pos_type = 'L'
            
            fee_i = pr_i * fee_rt
            tot_pl = tot_pl - fee_i
            exec_i = pd.DataFrame([{'DATE' : tm_i.date(), 'TIME' : tm_i.time(), 'POS_TYPE' : pos_type, 'POS' : pos, 'TR' : 'EL', 'PL' : 0, 'FEE' : fee_i}])         

            if len(exec) == 0:
                exec = exec_i
            else:
                exec = pd.concat([exec, exec_i])
                
            react_hold_tm = tm_i + timedelta(seconds = dt_react_hold)
            pr_ent = pr_i
            pr_h = pr_i
            pr_l = pr_i 
        
    return exec


In [270]:
dt_strtg_begin_buf = 30
dt_strtg_end_buf = 30
dp_lc = 10
dp_pt = 10
unit = 2.5
dp_pt_buf = 0
dt_react_obs = 60
dt_react_hold = 60

start_time = time.time()

perf_(dt_strtg_begin_buf, dt_strtg_end_buf, dp_lc, dp_pt, unit, dp_pt_buf, dt_react_obs, dt_react_hold, opt_atm)

end_time = time.time()
elapsed_time = end_time - start_time
print(f"Elapsed time: {elapsed_time:.2f} seconds")

Elapsed time: 113.27 seconds


In [397]:

# 각 거래: (DT, TR, PRC, QTY, POS, PL, FEE)
COL_TRADES = np.dtype([
    ('DT', np.int64)        #거래시간
    , ('TR', 'U2')          #거래유형(EL : Long 진입, ES : Short 진입, CL : 청산, LC : 손절, PT : 익절)
    , ('PR', np.float64)   #거래가격
    , ('QTY', np.int8)      #거래수량
    , ('POS', np.int8)      #잔고 
    , ('PL', np.float64)       #수익
    , ('FEE', np.float64)      #수수료
    ])

@njit
def run_strategy(dt, pr, f_rt, tr, trades, i):
    
    pos_old = 0
    pr_old = 0
    if i >= 0:
        pos_old = trades[i]['POS']
        pr_old  = trades[i]['PR']
   
    if ((tr == 'EL' and pos_old > 0) 
        or (tr == 'ES' and pos_old < 0) 
        or (tr in ['CL', 'LC', 'PT'] and pos_old == 0)) :
        return i 
    
    qty = 0
    pos = 0
    pl = 0
    fee = 0
    
    if pos_old != 0:        
        pl = (pr - pr_old) * np.sign(pos_old)    
        qty = -pos_old
        fee = pr * f_rt * abs(qty)
        
    HOLD_POS = 1
    if tr in ['EL', 'ES']:
        pos = HOLD_POS
        if tr == 'ES' :
            pos = -pos

        qty = qty + pos
        fee = fee + pr * f_rt * abs(pos)
        
    i = i + 1
    trades[i]['DT'] = dt
    trades[i]['TR'] = tr
    trades[i]['PR'] = pr
    trades[i]['QTY'] = qty
    trades[i]['POS'] = pos
    trades[i]['PL'] = pl
    trades[i]['FEE'] = fee    

    return i


@njit
def performance(strtg, ds_bgn, ds_end, lc, pt, dp_pt, ds_obs, ds_wt, f_rt = 0.003/100) :
    COL_DT = 0
    COL_DT_O = 1
    COL_DT_C = 2
    COL_FUT = 3
    COL_FUT_O = 4
    COL_FUT_H = 5
    COL_FUT_L = 6
    COL_DOPT_PRV = 7
    COL_DOPT = 8
    
    n = strtg.shape[0]
    trades = np.empty(n, dtype=COL_TRADES)
    
    i_tr_old = -1
    i_tr = i_tr_old
        
    dopt_tr = 0
    pr_tr_h = -np.inf
    pr_tr_l = np.inf
    dt_tr_obs = 0
    dt_tr_wt = 0

    is_strategy_on = False
    for i in range(n) :

        if i_tr_old != i_tr :
            pr_tr_h = -np.inf
            pr_tr_l = np.inf
            dopt_tr = 0
            
            if trades[i_tr]['TR'] in ['LC', 'PT', 'CL'] :
                dt_tr_obs = 0
                dt_tr_wt = 0
            
            i_tr_old = i_tr
        

        dt_i = strtg[i, COL_DT] 
        dt_o_i = strtg[i, COL_DT_O] 
        dt_c_i = strtg[i, COL_DT_C] 
        pr_i = strtg[i, COL_FUT] 
        pr_o_i = strtg[i, COL_FUT_O]    
        pr_h_i = strtg[i, COL_FUT_H]    
        pr_l_i = strtg[i, COL_FUT_L]    
        dopt_prv_i = strtg[i, COL_DOPT_PRV]    
        dopt_i = strtg[i, COL_DOPT]    

        if dt_i <= dt_o_i + ds_bgn :
            #장 개시 후 일정 시간 거래 스킵
            
            is_strategy_on = False
            continue
        elif dt_c_i - ds_end < dt_i :
            #장 종료 전, 일정 시간 거래 스킵
        
            if is_strategy_on == True:            
                #전략 실행 on 상태면, 청산 후 전략 실행 off
                i_tr = run_strategy(dt_i, pr_i, f_rt, 'CL', trades, i_tr)            

                is_strategy_on = False
            
            continue
        
        #전략 실행 on
        is_strategy_on = True
        
        tr_pos_i = 0
        tr_pr_i = 0
        tr_ls_i = 0
        
        if i_tr >= 0 :                
            trades_i = trades[i_tr]
            tr_pos_i = trades_i['POS']
            tr_pr_i = trades_i['PR']
            tr_ls_i = np.sign(tr_pos_i)
         
        #Loss Cut
        tr_pr_lc_i = tr_pr_i - tr_ls_i * lc
        if ((0 < tr_pos_i and pr_l_i <= tr_pr_lc_i)
            or (tr_pos_i < 0 and tr_pr_lc_i <= pr_h_i)) :
        
            i_tr = run_strategy(dt_i, tr_pr_lc_i, f_rt, 'LC', trades, i_tr)
            continue

        # Profit Take
        pr_tr_h = max(pr_tr_h, pr_h_i)
        pr_tr_l = min(pr_tr_l, pr_l_i)
                            
        tr_pr_pt_i = tr_pr_i + tr_ls_i * pt
        if ((0 < tr_pos_i and tr_pr_i + (pt + dp_pt) <= pr_tr_h and pr_i <= tr_pr_pt_i)
            or (tr_pos_i < 0 and pr_tr_l <= tr_pr_i - (pt + dp_pt) and tr_pr_pt_i <= pr_i)) :

            i_tr = run_strategy(dt_i, tr_pr_pt_i, f_rt, 'PT', trades, i_tr)
            continue
                
        #거래 대기 시간 이내 이면, 대응 스킵
        if dt_i <= dt_tr_obs or dt_i <= dt_tr_wt : 
            continue
  
        P = -1
        C = 1 
        if ((dopt_prv_i == P and dopt_i == C)
            or (dopt_prv_i == C and dopt_i == P)) :
            dt_tr_obs = dt_i + ds_obs
            dopt_tr = dopt_i
            continue
            
        # #Dominant유지 여부 체크
        if (dopt_tr == dopt_i and dopt_prv_i == dopt_i) :
            tr_i = ''
            if dopt_i == C :
                tr_i = 'EL'
            elif dopt_i == P :
                tr_i = 'ES'
                
            i_tr = run_strategy(dt_i, pr_i, f_rt, tr_i, trades, i_tr)
            dt_tr_wt = dt_i + ds_wt
            continue

    return trades[:i_tr]


In [401]:
ds_bgn = 3
ds_end = 30
lc = 10
pt = 10
unit = 2.5
dp_pt = 0
ds_obs = 60
ds_wt = 60

start_time = time.time()

# ===== 1. 파라미터 설정 =====
#전략 종료 시간 버퍼
ds_end = 10*60 + ds_end                     

#loss cut
lc = np.ceil(max(0, lc)/unit) * unit

#profit take
pt = np.ceil(max(0, pt)/unit) * unit

#profit take buffer
dp_pt = min(pt, (np.ceil(dp_pt)/unit) * unit)

ent_dp = 0.5         # 진입 가격 변동
clr_dp = 0.5         # 대응 가격 변동

#2. 전략 실행 데이터 생성
perf = opt_atm.copy()
perf = perf[['YRMO', 'DATE', 'TIME', 'FUT', 'FUT_O', 'FUT_H', 'FUT_L', 'OPT_DMNT_PRV', 'OPT_DMNT']]
perf['DT'] = pd.to_datetime(perf['DATE'] + ' ' + perf['TIME'], format = '%Y%m%d %H:%M:%S')
perf = perf.sort_values('DT', ascending = True).reset_index(drop=True)

perf['DATE'] = perf['DT'].dt.date
mkt_dt_o = perf.groupby('DATE')['DT'].min().reset_index()
mkt_dt_c = perf.groupby('DATE')['DT'].max().reset_index()
mkt_dt = pd.merge(mkt_dt_o, mkt_dt_c, how = 'inner', on = ['DATE'], suffixes=['_O', '_C'])
perf = pd.merge(perf, mkt_dt, how = 'inner', on = ['DATE'])

OPT_TO_NUM = {'P':-1, 'C':1}

strtg = perf[['DT', 'DT_O', 'DT_C', 'FUT', 'FUT_O', 'FUT_H', 'FUT_L', 'OPT_DMNT_PRV', 'OPT_DMNT']].copy()
    
strtg['DT'] = perf['DT'].astype(np.int64) // 10**9
strtg['DT_O'] = perf['DT_O'].astype(np.int64) // 10**9
strtg['DT_C'] = perf['DT_C'].astype(np.int64) // 10**9
strtg['OPT_DMNT'] = perf['OPT_DMNT'].map(OPT_TO_NUM)
strtg['OPT_DMNT_PRV'] = perf['OPT_DMNT_PRV'].map(OPT_TO_NUM)
strtg = strtg.to_numpy()

performance(strtg, ds_bgn, ds_end, lc, pt, dp_pt, ds_obs, ds_wt)

end_time = time.time()
elapsed_time = end_time - start_time
print(f"Elapsed time: {elapsed_time:.2f} seconds")

Elapsed time: 1.10 seconds


In [ ]:

# 각 거래: (DT, TR, PRC, QTY, POS, PL, FEE)
COL_TRADES = np.dtype([
    ('DT', np.int64)        #거래시간
    , ('TR', 'U2')          #거래유형(EL : Long 진입, ES : Short 진입, CL : 청산, LC : 손절, PT : 익절)
    , ('PR', np.float64)   #거래가격
    , ('QTY', np.int8)      #거래수량
    , ('POS', np.int8)      #잔고 
    , ('PL', np.float64)       #수익
    , ('FEE', np.float64)      #수수료
    ])

@njit
def run_strategy(dt, pr, f_rt, tr, trades, i):
    
    pos_old = 0
    pr_old = 0
    if i >= 0:
        pos_old = trades[i]['POS']
        pr_old  = trades[i]['PR']
   
    if ((tr == 'EL' and pos_old > 0) 
        or (tr == 'ES' and pos_old < 0) 
        or (tr in ['CL', 'LC', 'PT'] and pos_old == 0)) :
        return i 
    
    qty = 0
    pos = 0
    pl = 0
    fee = 0
    
    if pos_old != 0:        
        pl = (pr - pr_old) * np.sign(pos_old)    
        qty = -pos_old
        fee = pr * f_rt * abs(qty)
        
    HOLD_POS = 1
    if tr in ['EL', 'ES']:
        pos = HOLD_POS
        if tr == 'ES' :
            pos = -pos

        qty = qty + pos
        fee = fee + pr * f_rt * abs(pos)
        
    i = i + 1
    trades[i]['DT'] = dt
    trades[i]['TR'] = tr
    trades[i]['PR'] = pr
    trades[i]['QTY'] = qty
    trades[i]['POS'] = pos
    trades[i]['PL'] = pl
    trades[i]['FEE'] = fee    

    return i


@njit
def performance(strtg, ds_bgn, ds_end, lc, pt, dp_pt, ds_obs, ds_wt, f_rt = 0.003/100) :

    n = strtg.shape[0]
    trades = np.empty(n, dtype=COL_TRADES)
    
    i_tr_old = -1
    i_tr = i_tr_old
        
    dopt_tr = 'X'
    pr_tr_h = -np.inf
    pr_tr_l = np.inf
    dt_tr_obs = 0
    dt_tr_wt = 0

    is_strategy_on = False
    for i in range(n) :

        if i_tr_old != i_tr :
            pr_tr_h = -np.inf
            pr_tr_l = np.inf
            dopt_tr = 'X'
            
            if trades[i_tr]['TR'] in ['LC', 'PT', 'CL'] :
                dt_tr_obs = 0
                dt_tr_wt = 0
            
            i_tr_old = i_tr
        

        dt_i = strtg[i]['DT'] 
        dt_o_i = strtg[i]['DT_O'] 
        dt_c_i = strtg[i]['DT_C'] 
        pr_i = strtg[i]['FUT'] 
        pr_o_i = strtg[i]['FUT_O']    
        pr_h_i = strtg[i]['FUT_H']    
        pr_l_i = strtg[i]['FUT_L']    
        dopt_prv_i = strtg[i]['OPT_DMNT_PRV']    
        dopt_i = strtg[i]['OPT_DMNT']    

        if dt_i <= dt_o_i + ds_bgn :
            #장 개시 후 일정 시간 거래 스킵
            
            is_strategy_on = False
            continue
        elif dt_c_i - ds_end < dt_i :
            #장 종료 전, 일정 시간 거래 스킵
        
            if is_strategy_on == True:            
                #전략 실행 on 상태면, 청산 후 전략 실행 off
                i_tr = run_strategy(dt_i, pr_i, f_rt, 'CL', trades, i_tr)            

                is_strategy_on = False
            
            continue
        
        #전략 실행 on
        is_strategy_on = True
        
        tr_pos_i = 0
        tr_pr_i = 0
        tr_ls_i = 0
        
        if i_tr >= 0 :                
            trades_i = trades[i_tr]
            tr_pos_i = trades_i['POS']
            tr_pr_i = trades_i['PR']
            tr_ls_i = np.sign(tr_pos_i)
         
        #Loss Cut
        tr_pr_lc_i = tr_pr_i - tr_ls_i * lc
        if ((0 < tr_pos_i and pr_l_i <= tr_pr_lc_i)
            or (tr_pos_i < 0 and tr_pr_lc_i <= pr_h_i)) :
        
            i_tr = run_strategy(dt_i, tr_pr_lc_i, f_rt, 'LC', trades, i_tr)
            continue

        # Profit Take
        pr_tr_h = max(pr_tr_h, pr_h_i)
        pr_tr_l = min(pr_tr_l, pr_l_i)
                            
        tr_pr_pt_i = tr_pr_i + tr_ls_i * pt
        if ((0 < tr_pos_i and tr_pr_i + (pt + dp_pt) <= pr_tr_h and pr_i <= tr_pr_pt_i)
            or (tr_pos_i < 0 and pr_tr_l <= tr_pr_i - (pt + dp_pt) and tr_pr_pt_i <= pr_i)) :

            i_tr = run_strategy(dt_i, tr_pr_pt_i, f_rt, 'PT', trades, i_tr)
            continue
                
        #거래 대기 시간 이내 이면, 대응 스킵
        if dt_i <= dt_tr_obs or dt_i <= dt_tr_wt : 
            continue
        
        #Put/Call Cross Check
        if ((dopt_prv_i == 'P' and dopt_i == 'C')
            or (dopt_prv_i == 'C' and dopt_i == 'P')) :
            dt_tr_obs = dt_i + ds_obs
            dopt_tr = dopt_i
            print(dopt_tr)
            continue
        
        print(dopt_tr)

    #     # print(dopt_i)
    #     # print(dopt_prv_i)    

    #     # #Dominant유지 여부 체크
    #     # if (dopt_tr == dopt_i and dopt_prv_i == dopt_i) :
    #     #     tr_i = ''
    #     #     if dopt_i == 'C' :
    #     #         tr_i = 'EL'
    #     #     elif dopt_i == 'P' :
    #     #         tr_i = 'ES'
                
    #     #     i_tr = run_strategy(dt_i, pr_i, f_rt, tr_i, trades, i_tr)
    #     #     dt_tr_wt = dt_i + ds_wt
    #     #     continue

    # #     P = -1
    # #     C = 1 
    # #     if ((dopt_prv_i == P and dopt_i == C)
    # #         or (dopt_prv_i == C and dopt_i == P)) :
    # #         dt_tr_obs = dt_i + ds_obs
    # #         dopt_tr = dopt_i
    # #         continue
            
    # #     # #Dominant유지 여부 체크
    # #     if (dopt_tr == dopt_i and dopt_prv_i == dopt_i) :
    # #         tr_i = ''
    # #         if dopt_i == C :
    # #             tr_i = 'EL'
    # #         elif dopt_i == P :
    # #             tr_i = 'ES'
                
    # #         i_tr = run_strategy(dt_i, pr_i, f_rt, tr_i, trades, i_tr)
    # #         dt_tr_wt = dt_i + ds_wt
    # #         continue

    # # return trades[:i_tr]
    
print(performance(strtg, ds_bgn, ds_end, lc, pt, dp_pt, ds_obs, ds_wt))


ds_bgn = 30
ds_end = 30
lc = 10
pt = 10
unit = 2.5
dp_pt = 0
ds_obs = 60
ds_wt = 60

start_time = time.time()

# ===== 1. 파라미터 설정 =====
#전략 종료 시간 버퍼
ds_end = 10*60 + ds_end                     

#loss cut
lc = np.ceil(max(0, lc)/unit) * unit

#profit take
pt = np.ceil(max(0, pt)/unit) * unit

#profit take buffer
dp_pt = min(pt, (np.ceil(dp_pt)/unit) * unit)

ent_dp = 0.5         # 진입 가격 변동
clr_dp = 0.5         # 대응 가격 변동

#2. 전략 실행 데이터 생성
perf = opt_atm.copy()
perf = perf[['YRMO', 'DATE', 'TIME', 'FUT', 'FUT_O', 'FUT_H', 'FUT_L', 'OPT_DMNT_PRV', 'OPT_DMNT']]
perf['DT'] = pd.to_datetime(perf['DATE'] + ' ' + perf['TIME'], format = '%Y%m%d %H:%M:%S')
perf = perf.sort_values('DT', ascending = True).reset_index(drop=True)

perf['DATE'] = perf['DT'].dt.date
mkt_dt_o = perf.groupby('DATE')['DT'].min().reset_index()
mkt_dt_c = perf.groupby('DATE')['DT'].max().reset_index()
mkt_dt = pd.merge(mkt_dt_o, mkt_dt_c, how = 'inner', on = ['DATE'], suffixes=['_O', '_C'])
perf = pd.merge(perf, mkt_dt, how = 'inner', on = ['DATE'])

OPT_TO_NUM = {'P':-1, 'C':1}

# DataFrame을 NumPy 배열로 변환 (U10은 최대 10글자 문자열)
COL_STRTG = np.dtype([
    ('DT', np.int64)        
    , ('DT_O', np.int64)    
    , ('DT_C', np.int64)    
    , ('FUT', np.float64)   
    , ('FUT_O', np.float64)
    , ('FUT_H', np.float64)
    , ('FUT_L', np.float64)
    , ('OPT_DMNT_PRV', 'U1')
    , ('OPT_DMNT', 'U1')    
    ])

strtg = np.empty(len(perf), dtype=COL_STRTG)
strtg['DT'] = perf['DT'].astype(np.int64).values // 10**9
strtg['DT_O'] = perf['DT_O'].astype(np.int64).values // 10**9
strtg['DT_C'] = perf['DT_C'].astype(np.int64).values // 10**9
strtg['FUT'] = perf['FUT'].values 
strtg['FUT_O'] = perf['FUT_O'].values 
strtg['FUT_H'] = perf['FUT_H'].values 
strtg['FUT_L'] = perf['FUT_L'].values 
strtg['OPT_DMNT'] = perf['OPT_DMNT'].values
strtg['OPT_DMNT_PRV'] = perf['OPT_DMNT_PRV'].values

print(performance(strtg, ds_bgn, ds_end, lc, pt, dp_pt, ds_obs, ds_wt))

end_time = time.time()
elapsed_time = end_time - start_time
print(f"Elapsed time: {elapsed_time:.2f} seconds")

In [115]:
import pandas as pd
import numpy as np
from numba import njit
from numba.typed import List

# ===== 1. 샘플 DataFrame 생성 =====
data = {
    "date": ["2025-02-16"]*8 + ["2025-02-17"]*8,
    "time": ["10:00", "10:05", "10:10", "10:15", "10:20", "10:25", "10:30", "10:35"]*2,
    "WIN_PRV": ["C", "C", "P", "P", "C", "C", "P", "C"]*2,
    "WIN":    ["P", "P", "C", "C", "P", "P", "C", "C"]*2,
    "value":  [100, 105, 95, 90, 120, 130, 100, 115]*2,
    "high":   [105, 110, 100, 95, 125, 135, 105, 120]*2,
    "low":    [95, 100, 90, 85, 115, 125, 95, 110]*2,
    "open":   [100, 105, 95, 90, 120, 130, 100, 115]*2
}
df = pd.DataFrame(data)
df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'])
df = df.sort_values('datetime').reset_index(drop=True)

# ===== 2. 파라미터 설정 =====
dt   = pd.Timedelta(minutes=10)      # 신호 판단: t 시점에서 dt 후 WIN 상태 확인
dt_o = pd.Timedelta(minutes=10)      # 각 날짜 첫 시각부터 dt_o까지 거래 없음 (조건 11)
dt_c = pd.Timedelta(minutes=5)       # 각 날짜 마지막 시각 전 dt_c 시점부터 거래 없음 (조건 13)
wait_dt = pd.Timedelta(minutes=5)    # dominant 또는 stop loss 거래 후 거래 금지 시간
d = 10                               # stop loss 기준: 진입가와의 가격 차

# ===== 3. 신호 계산 (벡터화) =====
# 각 행의 t+dt 시점 계산
df['signal_time'] = df['datetime'] + dt
# merge_asof를 사용해 각 행에 대해 t+dt 시점의 WIN 값을 가져옴
future = pd.merge_asof(
    df[['datetime', 'WIN']],
    df[['datetime', 'WIN']],
    left_on='signal_time',
    right_on='datetime',
    direction='forward',
    suffixes=('', '_future')
)
df['WIN_future'] = future['WIN_future']

# P우세: t 시점에 WIN_PRV가 'C'이고 WIN가 'P'이며, t+dt 시점에도 WIN가 'P'
df['P_dominant'] = (df['WIN_PRV'] == 'C') & (df['WIN'] == 'P') & (df['WIN_future'] == 'P')
# C우세: t 시점에 WIN_PRV가 'P'이고 WIN가 'C'이며, t+dt 시점에도 WIN가 'C'
df['C_dominant'] = (df['WIN_PRV'] == 'P') & (df['WIN'] == 'C') & (df['WIN_future'] == 'C')

# ===== 4. Numba를 사용한 날짜별 시뮬레이션 함수 =====
# 거래 액션 코드는 다음과 같이 정의합니다:
# 0: STOP_LOSS_SELL, 1: STOP_LOSS_BUY, 2: SELL_ALL, 3: SELL_EXTRA,
# 4: BUY_ALL, 5: BUY_EXTRA, 6: EXIT
@njit
def simulate_day(times, values, p_dom, c_dom, dt_o_sec, dt_c_sec, wait_dt_sec, d):
    n = len(times)
    trades = List()  # 각 거래: (time, action_code, price, quantity)
    first_time = times[0]
    last_time  = times[n-1]
    start_trade_time = first_time + dt_o_sec
    end_trade_time   = last_time - dt_c_sec
    position = 1.0  # 초기 포지션: 롱 (+1)
    entry_price = values[0]
    last_trade_time_val = -1e9  # 초기 거래 제한 (매우 작은 값)
    dominant_executed = False
    
    for i in range(n):
        t = times[i]
        price = values[i]
        # 조건 11: 거래 시작 전은 패스
        if t < start_trade_time:
            continue
        # 조건 13: 거래 중단 시점 이후는 중단
        if t > end_trade_time:
            break
        # 거래 제한(wait) 기간이면 건너뜀
        if t < last_trade_time_val + wait_dt_sec:
            continue
        # 조건 8: 롱 상태에서 진입가보다 d 하락하면 stop loss 청산 (매도)
        if position > 0 and price <= entry_price - d:
            trades.append((t, 0, price, position))
            position = 0.0
            last_trade_time_val = t
            continue
        # 조건 9: 숏 상태에서 진입가보다 d 상승하면 stop loss 청산 (매수)
        if position < 0 and price >= entry_price + d:
            trades.append((t, 1, price, -position))
            position = 0.0
            last_trade_time_val = t
            continue
        # 조건 5,6: 아직 dominant 거래가 실행되지 않은 경우
        if not dominant_executed:
            # P우세: 롱 상태에서 p_dom True → 전량 매도 후 추가 매도 (포지션 -1)
            if p_dom[i] and position > 0:
                trades.append((t, 2, price, position))
                trades.append((t, 3, price, 1))
                position = -1.0
                entry_price = price
                dominant_executed = True
                last_trade_time_val = t
                continue
            # C우세: 숏 상태에서 c_dom True → 전량 매수 후 추가 매수 (포지션 +1)
            if c_dom[i] and position < 0:
                trades.append((t, 4, price, -position))
                trades.append((t, 5, price, 1))
                position = 1.0
                entry_price = price
                dominant_executed = True
                last_trade_time_val = t
                continue
    # 조건 12: 날짜의 마지막 시각 전 dt_c 시점에 포지션이 있으면, 해당 시점에 청산(종료)
    if position != 0:
        for i in range(n):
            if times[i] >= end_trade_time:
                trades.append((times[i], 6, values[i], abs(position)))
                break
    return trades

# ===== 5. Numba 시뮬레이션을 위한 데이터 변환 =====
# dt, dt_o, dt_c, wait_dt를 초 단위로 변환
dt_o_sec = dt_o.total_seconds()
dt_c_sec = dt_c.total_seconds()
wait_dt_sec = wait_dt.total_seconds()

# ===== 6. 날짜별 그룹화 후 Numba 시뮬레이션 실행 =====
all_trades = []
for date, group in df.groupby("date"):
    group = group.sort_values("datetime").reset_index(drop=True)
    # Convert datetime to Unix timestamp (초 단위)
    times = (group["datetime"].astype(np.int64) // 10**9).to_numpy().astype(np.float64)
    values = group["value"].to_numpy().astype(np.float64)
    p_dom = group["P_dominant"].to_numpy().astype(np.bool_)
    c_dom = group["C_dominant"].to_numpy().astype(np.bool_)
    trades = simulate_day(times, values, p_dom, c_dom, dt_o_sec, dt_c_sec, wait_dt_sec, d)
    # 각 거래에 날짜 정보 추가
    for tr in trades:
        all_trades.append((date, tr[0], tr[1], tr[2], tr[3]))

# ===== 7. 결과 DataFrame 생성 및 출력 =====
trade_df = pd.DataFrame(all_trades, columns=["date", "time_sec", "action", "price", "quantity"])
# action 코드 매핑 (예시)
action_map = {
    0: "STOP_LOSS_SELL",
    1: "STOP_LOSS_BUY",
    2: "SELL_ALL",
    3: "SELL_EXTRA",
    4: "BUY_ALL",
    5: "BUY_EXTRA",
    6: "EXIT"
}
trade_df["action"] = trade_df["action"].map(action_map)
# time_sec는 Unix timestamp (초); 이를 datetime으로 복원
trade_df["datetime"] = pd.to_datetime(trade_df["time_sec"], unit='s')
print(trade_df)


KeyError: 'Requested level (signal_time) does not match index name (None)'

In [194]:
import pandas as pd
import numpy as np
from numba import njit

# Step 1: pandas DataFrame 생성
data = {
    'A': [1, 2, 3, 4, 5],  # 숫자
    'B': ['A', 'B', 'C', 'A', 'A']  # 문자열
}
df = pd.DataFrame(data)

# 문자열 컬럼 'B'를 U20 (고정 길이 유니코드) 타입으로 변환
df['A'] = df['A'].astype(np.int64)
df['B'] = df['B'].astype('U2')

# Step 2: DataFrame을 numpy array로 변환
dd = df.to_numpy()  # 이제 dd는 2D numpy 배열

# 각 열의 데이터 유형 확인
for i in range(dd.shape[1]):
    print(f"Column {i} dtype: {dd[:, i].dtype}")
    
# # Step 3: numba의 @njit로 최적화된 함수 정의
# @njit
# def filter_rows(data):
#     result_A = []
#     result_B = []
#     for i in range(len(data)):
#         # 첫 번째 열(A열)은 숫자, 두 번째 열(B열)은 문자열
#         if data[i, 1] == 'A' or data[i, 1] == 'B':  # B열(문자열) 비교
#             result_A.append(data[i, 0])  # A열(숫자) 값 추가
#             result_B.append(data[i, 1])  # B열(문자열) 값 추가
#     return np.array(result_A), np.array(result_B)

# print(dd)
# # Step 4: numba 함수에 넘겨서 작업
# filtered_A, filtered_B = filter_rows(dd)

# # 결과 출력
# print("Filtered A:", filtered_A)
# print("Filtered B:", filtered_B)




Column 0 dtype: <U32
Column 1 dtype: <U32
Column 2 dtype: <U32


In [258]:
trades = np.zeros(3, dtype=COL_TRADES)
print(trades.shape)
print(trades)
trades[0]['TR'] = 'A'

for i in range(2):
    print(f"Column {i} dtype: {trades[i].dtype}")


# @njit
# def f(arr):
#     # 각 열의 데이터 유형 확인
#     return

# f(trades)

# # COL_TRADES = np.dtype([
# #     ('DT', np.int64)        #거래시간
# #     , ('TR', 'U2')          #거래유형(EL : Long 진입, ES : Short 진입, CL : 청산, LC : 손절, PT : 익절)
# #     , ('PR', np.float64)   #거래가격
# #     , ('QTY', np.int8)      #거래수량
# #     , ('POS', np.int8)      #잔고 
# #     , ('PL', np.float64)       #수익
# #     , ('FEE', np.float64)      #수수료
# #     ])

# AA = [
#     ('A', np.int64)        #거래시간
#     , ('B', 'U2')          #거래유형(EL : Long 진입, ES : Short 진입, CL : 청산, LC : 손절, PT : 익절)
# ]

import pandas as pd
import numpy as np
from numba import njit

# Step 1: pandas DataFrame 생성
data = {
    'A': [1, 2, 3, 4, 5],  # 숫자
    'B': ['A', 'B', 'C', 'A', 'A']  # 문자열
}
df = pd.DataFrame(data)

# 문자열 컬럼 'B'를 U20 (고정 길이 유니코드) 타입으로 변환
d = df.to_records(index = False)
print(d.shape)

print(d)
d[1].astype('U2')
      
      # d[0].astype(np.int64)   # 첫 번째 열을 int로 변환
# d[1].astype('U2') # 두 번째 열을 float로 변환

print(d[0].dtype)
print(d[1].dtype)

# @njit 
# def ff(dd):
#     print(dd)
#     return

# ff(d)


(3,)
[(0, '', 0., 0, 0, 0., 0.) (0, '', 0., 0, 0, 0., 0.)
 (0, '', 0., 0, 0, 0., 0.)]
Column 0 dtype: [('DT', '<i8'), ('TR', '<U2'), ('PR', '<f8'), ('QTY', 'i1'), ('POS', 'i1'), ('PL', '<f8'), ('FEE', '<f8')]
Column 1 dtype: [('DT', '<i8'), ('TR', '<U2'), ('PR', '<f8'), ('QTY', 'i1'), ('POS', 'i1'), ('PL', '<f8'), ('FEE', '<f8')]
(5,)
[(1, 'A') (2, 'B') (3, 'C') (4, 'A') (5, 'A')]


TypeError: Cannot cast scalar from dtype((numpy.record, [('A', '<i8'), ('B', 'O')])) to dtype('<U2') according to the rule 'unsafe'

In [370]:
import pandas as pd
import numpy as np
from numba import jit

# 예시 DataFrame 생성
df = pd.DataFrame({
    'numbers': [1, 2, 3],
    'strings': ['a', 'b', 'c']
})

# DataFrame을 NumPy의 기본 구조화된 배열로 변환하는 함수
def dataframe_to_structured_array(df):
    # DataFrame의 컬럼별로 dtype을 정의 (예: 'numbers'는 int, 'strings'는 str)
    dtype = [('numbers', 'i8'), ('strings', 'U10')]  # 'U10'은 최대 10글자의 문자열

    # np.empty()로 빈 구조화된 배열 생성
    structured_array = np.empty(len(df), dtype=dtype)

    # 각 컬럼을 구조화된 배열의 필드에 할당
    structured_array['numbers'] = df['numbers'].values
    structured_array['strings'] = df['strings'].values

    return structured_array

# DataFrame을 NumPy 구조화된 배열로 변환
structured_array = dataframe_to_structured_array(df)

# Numba JIT 데코레이터를 사용하여 함수 최적화
@jit(nopython=True)
def fnc(dd):
    # 문자열을 NumPy 배열로 초기화
    dopt = np.드ㅔㅅ(['X'], dtype='U10')  # NumPy 배열로 'X' 초기화
    for i in range(dd.shape[0]):
        dddd = dd[i]['strings']
        if i == 2:
            dopt[0] = dddd  # dopt 배열의 첫 번째 값을 업데이트
        print(i)
        print(dopt[0])  # 배열의 첫 번째 값 출력

fnc(structured_array)

# 결과 출력
print(structured_array)


AttributeError: module 'numpy' has no attribute '드ᅦᄉ'

In [266]:
@njit
def ff(arr) :
    for i in range(arr.shape[0]) :
        print(i)
        
        
ff

TypingError: Failed in nopython mode pipeline (step: nopython frontend)
non-precise type pyobject
During: typing of argument at /tmp/ipykernel_29483/1577018413.py (12)

File "../../../../../tmp/ipykernel_29483/1577018413.py", line 12:
<source missing, REPL/exec in use?>

During: Pass nopython_type_inference 

This error may have been caused by the following argument(s):
- argument 0: Cannot determine Numba type of <class 'pandas.core.frame.DataFrame'>
